# 01 — Data Acquisition

**Ziel**: Drei unabhängige Datenquellen für CycleSync beschaffen:
1. **Synthetischer Workout-Datensatz** — Generiert auf Basis publizierter sportwissenschaftlicher Parameter (Herzfrequenz, Performance über Zyklusphasen). Quelle für Bereich: Sims et al. (2023), McNulty et al. (2020).
2. **Synthetischer Zyklus-Tracking-Datensatz** — Symptom-, Schlaf-, BBT-Daten pro Zyklustag.
3. **Echte PubMed-Abstracts** — Wissenschaftliche Literatur als RAG-Wissensbasis (via NCBI E-utilities API).

**Warum synthetische Daten?** Echte zyklusgekoppelte Workout-Daten mit Phasen-Labels sind nicht öffentlich verfügbar. Wir generieren reproduzierbare Daten auf Basis publizierter physiologischer Parameter (Quellen werden dokumentiert).

Alle Daten landen in `data/raw/` und `data/pubmed_abstracts/`.

## Setup

In [12]:
# Wenn du in Colab arbeitest, das Repo klonen:
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB:
    # Repo klonen (nur beim ersten Run nötig)
    if not os.path.exists('/content/cyclesync'):
        !git clone https://github.com/carvacla/cyclesync_KI_project.git /content/cyclesync
    %cd /content/cyclesync/notebooks
    !pip install -q biopython pandas numpy scikit-learn

Cloning into '/content/cyclesync'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 37 (delta 6), reused 19 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 36.00 KiB | 12.00 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/cyclesync/notebooks
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.1 MB/s eta 0:00:00


In [13]:
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Pfade
DATA_RAW = Path('../data/raw')
ABSTRACTS_PATH = Path('../data/pubmed_abstracts')
DATA_RAW.mkdir(parents=True, exist_ok=True)
ABSTRACTS_PATH.mkdir(parents=True, exist_ok=True)

print(f'Raw data: {DATA_RAW.absolute()}')
print(f'Abstracts: {ABSTRACTS_PATH.absolute()}')

Raw data: /content/cyclesync/notebooks/../data/raw
Abstracts: /content/cyclesync/notebooks/../data/pubmed_abstracts


## 1. Synthetischer Zyklus-Tracking-Datensatz

Wir simulieren **500 Nutzerinnen über je 3 Zyklen** (~28 Tage), basierend auf publizierten Mustern:

- **Menstruation (Tag 1–5)**: Höhere Müdigkeit, niedrigere Schlafqualität, leichte BBT-Senkung
- **Frühe Follikelphase (Tag 6–10)**: Energie steigt, wenige Symptome, BBT niedrig
- **Späte Follikelphase / Ovulation (Tag 11–16)**: Höchste Energie, Schlaf gut, BBT-Sprung
- **Frühe Lutealphase (Tag 17–22)**: BBT erhöht, Schlaf okay
- **Späte Lutealphase / PMS (Tag 23–28)**: Symptome wie Bloating, Stimmung niedrig, Schlafqualität sinkt

Quellen für Parameter: Schmalenberger et al. (2021), Carmichael et al. (2021).

In [14]:
def get_phase(day_in_cycle: int) -> str:
    '''Mappt Zyklustag auf Phase.'''
    if day_in_cycle <= 5:
        return 'menstruation'
    elif day_in_cycle <= 13:
        return 'follicular'
    elif day_in_cycle <= 16:
        return 'ovulation'
    else:
        return 'luteal'


def simulate_day(user_id: int, cycle_num: int, day: int, user_baseline: dict) -> dict:
    '''Simuliert einen Tag im Zyklus mit Phasen-typischen Mustern.'''
    phase = get_phase(day)

    # BBT (Basal Body Temperature) — typisches biphasisches Muster
    if phase in ('menstruation', 'follicular'):
        bbt = 36.4 + np.random.normal(0, 0.15)
    elif phase == 'ovulation':
        bbt = 36.55 + np.random.normal(0, 0.15)
    else:  # luteal
        bbt = 36.75 + np.random.normal(0, 0.18)

    # Schlaf
    sleep_baselines = {
        'menstruation': (6.8, 5),
        'follicular': (7.5, 7),
        'ovulation': (7.6, 8),
        'luteal': (7.0, 6),
    }
    sleep_mean, qual_mean = sleep_baselines[phase]
    sleep_hours = np.clip(np.random.normal(sleep_mean, 0.8), 4, 10)
    sleep_quality = int(np.clip(np.random.normal(qual_mean, 1.5), 1, 10))

    # Symptome (Wahrscheinlichkeiten je nach Phase)
    symptom_probs = {
        'menstruation': {'cramps': 0.7, 'fatigue': 0.6, 'mood_low': 0.4, 'headache': 0.3, 'bloating': 0.5, 'tender_breasts': 0.1},
        'follicular':   {'cramps': 0.05, 'fatigue': 0.15, 'mood_low': 0.1, 'headache': 0.1, 'bloating': 0.1, 'tender_breasts': 0.05},
        'ovulation':    {'cramps': 0.15, 'fatigue': 0.1, 'mood_low': 0.05, 'headache': 0.15, 'bloating': 0.2, 'tender_breasts': 0.2},
        'luteal':       {'cramps': 0.2, 'fatigue': 0.5, 'mood_low': 0.45, 'headache': 0.3, 'bloating': 0.55, 'tender_breasts': 0.5},
    }
    probs = symptom_probs[phase]
    symptoms = [s for s, p in probs.items() if np.random.random() < p]
    if not symptoms:
        symptoms = ['none']

    # Resting Heart Rate (RHR) — leicht erhöht in Lutealphase
    rhr_offset = {'menstruation': 1, 'follicular': 0, 'ovulation': 0, 'luteal': 3}[phase]
    rhr = int(np.clip(np.random.normal(user_baseline['rhr'] + rhr_offset, 3), 45, 90))

    return {
        'user_id': user_id,
        'cycle_num': cycle_num,
        'day_in_cycle': day,
        'phase': phase,
        'bbt_celsius': round(bbt, 2),
        'sleep_hours': round(sleep_hours, 1),
        'sleep_quality': sleep_quality,
        'symptoms': ';'.join(symptoms),
        'resting_hr': rhr,
        'age': user_baseline['age'],
        'fitness_level': user_baseline['fitness'],
    }


# Generierung
N_USERS = 500
N_CYCLES = 3
CYCLE_LENGTH = 28

rows = []
for uid in range(N_USERS):
    # Baseline-Profil pro User
    baseline = {
        'age': int(np.clip(np.random.normal(30, 8), 18, 50)),
        'rhr': int(np.clip(np.random.normal(65, 8), 50, 85)),
        'fitness': np.random.choice(['beginner', 'intermediate', 'advanced'], p=[0.3, 0.5, 0.2]),
    }
    for c in range(N_CYCLES):
        for d in range(1, CYCLE_LENGTH + 1):
            rows.append(simulate_day(uid, c, d, baseline))

cycle_df = pd.DataFrame(rows)
cycle_df.to_csv(DATA_RAW / 'cycle_tracking.csv', index=False)
print(f'Cycle tracking dataset: {cycle_df.shape}')
cycle_df.head()

Cycle tracking dataset: (42000, 11)


,user_id,cycle_num,day_in_cycle,phase,bbt_celsius,sleep_hours,sleep_quality,symptoms,resting_hr,age,fitness_level
0,0,0,1,menstruation,36.23,7.1,5,fatigue;bloating,67,33,intermediate
1,0,0,2,menstruation,36.31,6.1,1,cramps;mood_low;headache;bloating,66,33,intermediate
2,0,0,3,menstruation,36.31,7.3,8,cramps;mood_low;headache,65,33,intermediate
3,0,0,4,menstruation,36.31,7.6,5,cramps;fatigue;headache,62,33,intermediate
4,0,0,5,menstruation,36.21,7.7,9,cramps,67,33,intermediate


## 2. Synthetischer Workout-Datensatz

Wir generieren Workout-Logs mit Phasen-abhängigen Performance-Mustern. Pro User ~20 Workouts über die Zyklen verteilt.

Physiologische Annahmen (basierend auf McNulty et al. 2020 Meta-Analyse):
- **Follikelphase**: Beste Kraft- und HIIT-Performance
- **Lutealphase**: Höhere Herzfrequenz bei gleicher Belastung, schlechtere thermische Regulation
- **Menstruation**: Reduzierte Performance, höhere wahrgenommene Anstrengung (RPE)

In [15]:
SPORTS = ['running', 'cycling', 'strength', 'yoga', 'hiit', 'walking']

phase_performance = {
    'menstruation': {'hr_factor': 1.05, 'rpe_offset': 1.0, 'recovery_factor': 1.2},
    'follicular':   {'hr_factor': 1.00, 'rpe_offset': 0.0, 'recovery_factor': 1.0},
    'ovulation':    {'hr_factor': 1.00, 'rpe_offset': -0.3, 'recovery_factor': 0.95},
    'luteal':       {'hr_factor': 1.08, 'rpe_offset': 0.7, 'recovery_factor': 1.15},
}

workout_rows = []
for uid, group in cycle_df.groupby('user_id'):
    user_meta = group.iloc[0]
    age = user_meta['age']
    fitness = user_meta['fitness_level']
    max_hr = 220 - age
    fitness_multiplier = {'beginner': 0.95, 'intermediate': 1.0, 'advanced': 1.05}[fitness]

    # Wähle zufällig 20 Tage aus den 84 Zyklustagen für Workouts
    workout_days = group.sample(n=20, random_state=uid)

    for _, day in workout_days.iterrows():
        phase = day['phase']
        sport = np.random.choice(SPORTS)
        duration = int(np.clip(np.random.normal(45, 15), 15, 120))

        # Intensitäts-Target (1–10)
        intensity_target = int(np.clip(np.random.choice([3, 4, 5, 6, 7, 8]), 1, 10))

        # Tatsächliche durchschnittliche HR während Workout
        base_hr_pct = 0.55 + (intensity_target / 10) * 0.30  # 55–85% von max
        avg_hr = int(max_hr * base_hr_pct * phase_performance[phase]['hr_factor'] / fitness_multiplier)
        avg_hr = int(np.clip(avg_hr + np.random.normal(0, 4), 90, max_hr))

        # RPE (Rating of Perceived Exertion, 1–10)
        rpe = intensity_target + phase_performance[phase]['rpe_offset'] + np.random.normal(0, 0.5)
        rpe = float(np.clip(round(rpe, 1), 1, 10))

        # Recovery-Stunden
        base_recovery = duration * (intensity_target / 10) * 1.2
        recovery_h = base_recovery * phase_performance[phase]['recovery_factor']
        recovery_h = int(np.clip(recovery_h + np.random.normal(0, 4), 6, 72))

        # Wetter (für Realismus / Feature-Diversität)
        temp_c = round(np.random.normal(15, 8), 1)

        workout_rows.append({
            'user_id': uid,
            'cycle_num': day['cycle_num'],
            'day_in_cycle': day['day_in_cycle'],
            'phase': phase,
            'sport': sport,
            'duration_min': duration,
            'intensity_target': intensity_target,
            'avg_hr_bpm': avg_hr,
            'rpe': rpe,
            'recovery_hours': recovery_h,
            'temp_c': temp_c,
            'age': age,
            'fitness_level': fitness,
        })

workout_df = pd.DataFrame(workout_rows)
workout_df.to_csv(DATA_RAW / 'workouts.csv', index=False)
print(f'Workout dataset: {workout_df.shape}')
workout_df.head()

Workout dataset: (10000, 13)


,user_id,cycle_num,day_in_cycle,phase,sport,duration_min,intensity_target,avg_hr_bpm,rpe,recovery_hours,temp_c,age,fitness_level
0,0,1,3,menstruation,walking,18,7,143,7.8,15,20.7,33,intermediate
1,0,1,13,follicular,strength,15,4,127,4.2,6,10.0,33,intermediate
2,0,1,16,ovulation,strength,42,8,141,8.1,41,1.5,33,intermediate
3,0,1,23,luteal,walking,61,5,131,5.9,31,17.7,33,intermediate
4,0,0,23,luteal,hiit,45,6,145,6.9,34,15.6,33,intermediate


## 3. Empfehlungs-Labels generieren

Aus den simulierten Daten leiten wir per Sportwissenschafts-Regeln das **Label `recommended_intensity`** (low/moderate/high) ab. Dieses Label ist das Ziel des ML-Modells.

In [16]:
def recommend_intensity(row) -> str:
    '''Heuristische Empfehlungs-Regel basierend auf Sportwissenschaft.'''
    phase = row['phase']
    sleep_quality = row['sleep_quality']
    symptoms = row['symptoms'].split(';')
    high_burden_symptoms = {'cramps', 'fatigue', 'headache'}
    symptom_burden = len(set(symptoms) & high_burden_symptoms)

    # Schlechter Schlaf ODER hohe Symptomlast → konservativ
    if sleep_quality <= 4 or symptom_burden >= 2:
        return 'low'

    # Phase-basierte Standardempfehlung
    if phase == 'menstruation':
        return 'low' if symptom_burden >= 1 else 'moderate'
    elif phase == 'follicular':
        return 'high'
    elif phase == 'ovulation':
        return 'high'
    elif phase == 'luteal':
        # Frühe vs. späte Lutealphase
        if row['day_in_cycle'] >= 23:
            return 'low' if sleep_quality < 7 else 'moderate'
        return 'moderate'
    return 'moderate'


cycle_df['recommended_intensity'] = cycle_df.apply(recommend_intensity, axis=1)
cycle_df.to_csv(DATA_RAW / 'cycle_tracking.csv', index=False)
print('Label-Verteilung:')
print(cycle_df['recommended_intensity'].value_counts())

Label-Verteilung:
recommended_intensity
low         20206
high        14736
moderate     7058
Name: count, dtype: int64


## 4. PubMed-Abstracts via NCBI E-utilities

Wir holen echte wissenschaftliche Abstracts zu zyklusbasiertem Training. Diese bilden die **Wissensbasis für die RAG-Pipeline**.

In [17]:
from Bio import Entrez

# WICHTIG: Trage deine E-Mail ein (NCBI verlangt das für API-Nutzung)
Entrez.email = 'carvacla@students.zhaw.ch'

SEARCH_QUERIES = [
    'menstrual cycle exercise performance',
    'luteal phase training strength',
    'follicular phase athletic performance',
    'menstrual cycle recovery sleep',
    'hormonal fluctuations sports women',
    'premenstrual syndrome exercise',
    'estrogen progesterone exercise physiology',
    'menstrual cycle injury risk female athletes',
]

MAX_PER_QUERY = 50  # ergibt ~300-400 Abstracts total

all_pmids = set()

for query in SEARCH_QUERIES:
    handle = Entrez.esearch(db='pubmed', term=query, retmax=MAX_PER_QUERY, sort='relevance')
    record = Entrez.read(handle)
    handle.close()
    pmids = record['IdList']
    all_pmids.update(pmids)
    print(f'Query "{query}": {len(pmids)} Treffer')
    time.sleep(0.4)  # Rate-Limiting (3 req/sec ohne API-Key)

print(f'\nGesamt unique PMIDs: {len(all_pmids)}')

Query "menstrual cycle exercise performance": 50 Treffer
Query "luteal phase training strength": 50 Treffer
Query "follicular phase athletic performance": 50 Treffer
Query "menstrual cycle recovery sleep": 32 Treffer
Query "hormonal fluctuations sports women": 50 Treffer
Query "premenstrual syndrome exercise": 50 Treffer
Query "estrogen progesterone exercise physiology": 34 Treffer
Query "menstrual cycle injury risk female athletes": 50 Treffer

Gesamt unique PMIDs: 318


In [18]:
def fetch_abstracts(pmids: list, batch_size: int = 50) -> list:
    '''Holt Abstracts in Batches.'''
    abstracts = []
    for i in range(0, len(pmids), batch_size):
        batch = pmids[i:i + batch_size]
        handle = Entrez.efetch(db='pubmed', id=batch, rettype='abstract', retmode='xml')
        records = Entrez.read(handle)
        handle.close()

        for article in records['PubmedArticle']:
            try:
                pmid = str(article['MedlineCitation']['PMID'])
                art = article['MedlineCitation']['Article']
                title = str(art.get('ArticleTitle', ''))

                # Abstract zusammensetzen (kann mehrere Sektionen haben)
                abstract_parts = art.get('Abstract', {}).get('AbstractText', [])
                if isinstance(abstract_parts, list):
                    abstract = ' '.join(str(p) for p in abstract_parts)
                else:
                    abstract = str(abstract_parts)

                # Jahr
                year = None
                try:
                    year = int(art['Journal']['JournalIssue']['PubDate'].get('Year', 0))
                except (KeyError, ValueError, TypeError):
                    pass

                # Autoren
                authors = []
                for a in art.get('AuthorList', []):
                    if 'LastName' in a:
                        authors.append(f"{a.get('LastName', '')} {a.get('Initials', '')}".strip())

                if abstract and len(abstract) > 100:
                    abstracts.append({
                        'pmid': pmid,
                        'title': title,
                        'abstract': abstract,
                        'year': year,
                        'authors': authors[:5],
                    })
            except Exception as e:
                print(f'Skip article (parsing error): {e}')
                continue

        print(f'Batch {i // batch_size + 1}: {len(abstracts)} Abstracts so far')
        time.sleep(0.4)

    return abstracts

abstracts = fetch_abstracts(list(all_pmids))
print(f'\nTotal Abstracts mit Volltext: {len(abstracts)}')

Batch 1: 49 Abstracts so far
Batch 2: 96 Abstracts so far
Batch 3: 145 Abstracts so far
Batch 4: 193 Abstracts so far
Batch 5: 242 Abstracts so far
Batch 6: 290 Abstracts so far
Batch 7: 307 Abstracts so far

Total Abstracts mit Volltext: 307


In [20]:
# Speichern als JSONL
output_file = ABSTRACTS_PATH / 'pubmed_abstracts.jsonl'
with open(output_file, 'w', encoding='utf-8') as f:
    for a in abstracts:
        f.write(json.dumps(a, ensure_ascii=False) + '\n')

print(f'Gespeichert: {output_file}')
print(f'Beispiel-Eintrag:\n')
print(json.dumps(abstracts[0], indent=2, ensure_ascii=False)[:1000])

Gespeichert: ../data/pubmed_abstracts/pubmed_abstracts.jsonl
Beispiel-Eintrag:

{
  "pmid": "7752873",
  "title": "Effects of menstrual cycle phase on athletic performance.",
  "abstract": "The purpose of this study was to examine the effects of menstrual cycle phase on four selected indices of athletic performance: aerobic capacity, anaerobic capacity, isokinetic strength, and high intensity endurance. Sixteen eumenorrheic women (VO2max > or = 50 ml.kg-1.min-1) were tested during the early follicular (F) and midluteal (L) phases of the menstrual cycle. Cycle phases were confirmed by serum estradiol and progesterone assays. No significant differences were observed between F and L tests in weight, percent body fat, sum of skinfolds, hemoglobin concentration, hematocrit, maximum heart rate, maximum minute ventilation, maximum respiratory exchange ratio, anaerobic performance, endurance time to fatigue (at 90% of VO2max), or isokinetic strength of knee flexion and extension. Both absolute

## Zusammenfassung

Drei Datenquellen liegen jetzt in `data/`:
1. `data/raw/cycle_tracking.csv` — Zyklus-Tracking (500 Userinnen × 3 Zyklen × 28 Tage)
2. `data/raw/workouts.csv` — Workout-Logs (~10'000 Workouts)
3. `data/pubmed_abstracts/pubmed_abstracts.jsonl` — Echte PubMed-Abstracts

Weiter mit `02_eda.ipynb`.